# Домашняя бухгалтерия с нейросетью — Этап 2 (модели)
Блокнот для Google Colab. **Runtime → Change runtime type → T4 GPU**, затем **Run all**.
Обучает с нуля (без предобученных весов): классификатор категорий (char-CNN) и бейзлайн TF-IDF+LogReg, генерирует синтетическую историю трат, обучает LSTM-прогноз и два бейзлайна, экспортирует модели в ONNX и проверяет совпадение с PyTorch.

В конце блокнот скачивает `stage2_outputs.zip` (метрики, графики, модели) — пришлите его или его содержимое.

## 1. Код проекта
Загрузите `home-budget-stage1.zip` (архив проекта) в файлы Colab или укажите `REPO_URL`. Если после этапа 1 сессия Colab ещё жива и данные лежат в `home-budget/ml/data/processed/`, их можно использовать повторно.

In [ ]:
import os, subprocess, zipfile, glob
REPO_URL = ""          # например: "https://github.com/<user>/home-budget.git"
cwd = os.getcwd()
base = os.path.dirname(cwd) if os.path.basename(cwd) == "home-budget" else cwd
zips = glob.glob(os.path.join(base, "home-budget*.zip"))
if REPO_URL and not os.path.isdir(os.path.join(base, "home-budget")):
    subprocess.run(["git", "clone", REPO_URL, os.path.join(base, "home-budget")], check=True)
elif zips:
    newest = max(zips, key=os.path.getmtime)          # самый свежий загруженный архив
    z = zipfile.ZipFile(newest)
    # архив может быть с папкой home-budget/ наверху или без неё — поддерживаем оба варианта;
    # код перезаписывается, скачанные данные в ml/data не трогаются
    z.extractall(base if all(n.startswith("home-budget/") for n in z.namelist()) else os.path.join(base, "home-budget"))
    print("распакован архив:", os.path.basename(newest))
elif not os.path.isdir(os.path.join(base, "home-budget")):
    raise SystemExit("Укажите REPO_URL или загрузите zip проекта")
os.chdir(os.path.join(base, "home-budget"))
print(os.getcwd(), os.listdir("."))


In [ ]:
!pip -q install duckdb pyarrow pandas scikit-learn pyyaml huggingface_hub pytest onnx onnxscript onnxruntime matplotlib

## 2. Тесты кода (в том числе моделей и ONNX-экспорта — здесь есть PyTorch)

In [ ]:
!python -m pytest -q

## 3. Данные Этапа 1
`REBUILD = True` пересобирает датасет текущим кодом (рекомендуется: в архив вошли правки после 4-го прогона). Скачанные файлы повторно не качаются. Если сессия новая, шаг скачивает Open*Facts заново (несколько минут).
Для проверки без скачивания поставьте `SEED_ONLY = True` — получится маленький датасет только из словаря (метрики будут заниженными и не годятся для отчёта).

In [ ]:
import os
REBUILD = True
SEED_ONLY = False
os.environ["REBUILD"] = "1" if REBUILD else "0"
os.environ["SEED_ONLY"] = "1" if SEED_ONLY else "0"

In [ ]:
%%bash
if [ "$SEED_ONLY" = "1" ]; then
  python -m ml.scripts.02_build_category_dataset --seed-only
elif [ -f ml/data/processed/category_dataset.parquet ] && [ "$REBUILD" != "1" ]; then
  echo "category_dataset.parquet уже есть - пропускаем сборку"
else
  python -m ml.scripts.01_download && python -m ml.scripts.02_build_category_dataset
fi

### Реальный тест (D6)
Загрузите `real_test_labeled.csv` (разметка реальных чеков; в архиве его нет — реальные данные не коммитятся). Он **не** используется при обучении, только для оценки.

In [ ]:
import os, shutil
from google.colab import files
target = "ml/data/processed/real_test_labeled.csv"
if not os.path.exists(target):
    up = files.upload()
    for name in up:
        os.makedirs(os.path.dirname(target), exist_ok=True)
        shutil.move(name, target)
        break
print("real_test_labeled.csv:", os.path.exists(target))

In [ ]:
import pandas as pd
from ml.tableio import read_table
df = read_table("ml/data/processed/category_dataset.parquet")
print(df.shape); print(df["split"].value_counts().to_dict()); print(df["source"].value_counts().to_dict())
print("классов:", df["subcategory"].nunique(), " минимум train на класс:", df[df.split=="train"]["subcategory"].value_counts().min())

## 4. Классификатор категорий
Сначала быстрая проверка (1 эпоха на малой выборке) — чтобы ошибка не обнаружилась через 20 минут.

In [ ]:
!python -m ml.train_classifier --smoke

In [ ]:
!python -m ml.train_classifier

In [ ]:
!python -m ml.export_onnx --only classifier
!python -m ml.evaluate classifier

In [ ]:
import json
m = json.load(open("ml/reports/metrics_classifier.json", encoding="utf-8"))
for k in ("test_synth", "real_test", "real_test_with_threshold_0.5"):
    if k in m["metrics"]: print(k, {a: round(b, 3) for a, b in m["metrics"][k].items() if isinstance(b, float)})
print("бейзлайн real_test:", {a: round(b, 3) for a, b in m["baseline"].get("real_test", {}).items() if isinstance(b, float)})
print("выбор для продакшена:", m.get("production_choice"), "-", m.get("production_choice_reason"))
print("ONNX МБ:", m.get("onnx_size_mb"), " латентность 50 позиций, мс:", m.get("latency_ms_50_items"), " расхождение с torch:", m.get("onnx_parity_max_abs_diff"))
if "ensemble" in m: print("ансамбль real_test:", {a: round(b, 3) for a, b in m["ensemble"]["real_test"].items() if isinstance(b, float)})
for name in ("holdout_receipts", "without_dictionary_overlap", "without_cafe"):
    row = lambda d: {a: round(b, 3) for a, b in d.get(name, {}).items() if a in ("n", "acc_cat", "macro_f1_cat")}
    print(name, "| нейросеть", row(m["metrics"]["real_test_views"]), "| бейзлайн", row(m["baseline"].get("real_test_views", {})))
print("лёгкий бейзлайн:", {k: m["baseline"].get(k) for k in ("features", "size_mb", "latency_ms_50_items", "numpy_vs_sklearn_argmax_agreement")})


## 5. LSTM-прогноз — НЕОБЯЗАТЕЛЬНЫЙ эксперимент (по умолчанию выключен)
**Решение от 2026-10-28:** приложение не показывает прогноз как факт. Вместо него — история самого пользователя (траты по месяцам, типичный месяц, темп относительно прошлых месяцев) и его цели-лимиты (`app/services/history.py`, `goals.py`). Обучение на синтетике для этого не нужно.

Ячейки ниже оставлены как документированный эксперимент (честное сравнение LSTM с бейзлайнами на синтетике). Чтобы запустить, поставьте `RUN_LSTM_EXPERIMENT = True`. Результат **не** попадает в приложение.

In [ ]:
RUN_LSTM_EXPERIMENT = False   # True — обучить LSTM на синтетической истории (около 10–20 минут на T4)
import subprocess, sys
def run(*args):
    subprocess.run([sys.executable, "-m", *args], check=True)
if RUN_LSTM_EXPERIMENT:
    run("ml.scripts.05_generate_transactions")
    run("ml.train_forecaster", "--smoke")
    run("ml.train_forecaster")
    run("ml.export_onnx", "--only", "forecaster")
    run("ml.evaluate", "forecaster")
else:
    print("LSTM-эксперимент пропущен (по решению: прогноз как факт убран).")

## 6. Установка артефактов и итоговые проверки

In [ ]:
!python -m ml.export_onnx --install
!ls -la app/ml_artifacts
!python -m pytest -q tests/test_ml_artifacts.py

In [ ]:
import zipfile, os
from google.colab import files
out = "stage2_outputs.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for d in ("ml/reports", "app/ml_artifacts"):
        for root, _, names in os.walk(d):
            for n in names: z.write(os.path.join(root, n))
print(out, round(os.path.getsize(out) / 1e6, 2), "МБ")
files.download(out)